# 05 — Churn Prediction & Retention Scoring
**Module 3 — Who Is About to Stop Paying?**

Builds two complementary models:
1. **Survival analysis** (Kaplan-Meier + Cox Proportional Hazard) — models *when* a customer is likely to churn
2. **30-day binary classifier** (XGBoost) — predicts *whether* an active customer will churn in the next 30 days

Output: a daily risk score per customer for field agent or call centre action.

---

## Churn Definition
A customer is **churned** when they:
- Miss payments for 7+ consecutive days (product lockout threshold), OR
- Have been locked out 3+ times in a single contract year

## ⚠️ Simulated Data Disclaimer
The payment event ledger is synthetically generated using GOGLA PERFORM benchmark distributions. See Notebook 04 for full disclaimer.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
np.random.seed(42)

ROOT   = Path("..")
SIM    = ROOT / "data" / "simulated"
FIGS   = ROOT / "outputs" / "figures"
MODELS = ROOT / "outputs" / "models"

print("Environment ready.")

## 5.1 Simulate Payment Event Ledger

In [ ]:
N_CUSTOMERS = 8_000
rng = np.random.default_rng(42)

contract_months     = rng.choice([12, 18, 24, 30, 36], N_CUSTOMERS, p=[0.1, 0.2, 0.35, 0.2, 0.15])
contract_days       = contract_months * 30
downpayment_pct     = rng.beta(2, 5, N_CUSTOMERS) * 0.35
is_urban            = rng.binomial(1, 0.45, N_CUSTOMERS)
wealth_index        = rng.normal(0, 1, N_CUSTOMERS)
mobile_money_freq   = rng.choice([0,1,2,3], N_CUSTOMERS, p=[0.15, 0.30, 0.35, 0.20])

# Churn hazard (higher = churn sooner)
churn_hazard = (
    0.002
    + 0.005 * (downpayment_pct < 0.05).astype(float)
    - 0.001 * mobile_money_freq
    - 0.0008 * wealth_index
    - 0.0005 * is_urban
).clip(0.0005, 0.015)

# Days to churn (exponential survival)
days_to_churn_raw = rng.exponential(1 / churn_hazard)
observed_duration  = np.minimum(days_to_churn_raw, contract_days).astype(int)
churned            = (days_to_churn_raw <= contract_days).astype(int)

# ── Payment behaviour features ────────────────────────────────────────────────
days_since_last_pmt  = rng.integers(1, 30, N_CUSTOMERS) * (1 + 2 * churned)
payment_regularity   = rng.exponential(3, N_CUSTOMERS) + 2 * churned      # std dev of intervals
pmt_vs_schedule      = rng.normal(1.0, 0.2, N_CUSTOMERS) - 0.3 * churned  # ratio: actual/expected
days_locked          = rng.integers(0, 5, N_CUSTOMERS) + 5 * churned
contract_age_days    = observed_duration

survival_df = pd.DataFrame({
    "customer_id":           [f"CUS{i:06d}" for i in range(N_CUSTOMERS)],
    "duration_days":         observed_duration,
    "churned":               churned,
    "downpayment_pct":       downpayment_pct.round(4),
    "contract_months":       contract_months,
    "is_urban":              is_urban,
    "wealth_index":          wealth_index.round(3),
    "mobile_money_freq":     mobile_money_freq,
    "days_since_last_pmt":   days_since_last_pmt,
    "payment_regularity":    payment_regularity.round(3),
    "pmt_vs_schedule":       pmt_vs_schedule.round(3),
    "days_locked":           days_locked,
})

survival_df.to_csv(SIM / "churn_customer_ledger.csv", index=False)

print(f"{N_CUSTOMERS:,} customers | Churn rate: {churned.mean():.1%}")
survival_df.head()

## 5.2 Kaplan-Meier Survival Curves

In [ ]:
survival_df["dp_high"] = (survival_df["downpayment_pct"] > 0.10).astype(int)

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(9, 5))

for group, label, color in [(1, "Downpayment > 10%", "#2196F3"), (0, "Downpayment ≤ 10%", "#EF5350")]:
    mask = survival_df["dp_high"] == group
    kmf.fit(survival_df.loc[mask, "duration_days"], survival_df.loc[mask, "churned"], label=label)
    kmf.plot_survival_function(ax=ax, color=color, ci_show=True)

ax.set_xlabel("Days Since Onboarding")
ax.set_ylabel("Survival Probability (Active Customer)")
ax.set_title("Kaplan-Meier Survival Curves by Downpayment Group", fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / "05_km_survival.png", dpi=150, bbox_inches="tight")
plt.show()

# Log-rank test
results_lr = logrank_test(
    survival_df.loc[survival_df.dp_high==1, "duration_days"],
    survival_df.loc[survival_df.dp_high==0, "duration_days"],
    event_observed_A=survival_df.loc[survival_df.dp_high==1, "churned"],
    event_observed_B=survival_df.loc[survival_df.dp_high==0, "churned"],
)
print(f"Log-rank test p-value: {results_lr.p_value:.4f}")

## 5.3 Cox Proportional Hazard Model

In [ ]:
cox_features = [
    "duration_days", "churned",
    "downpayment_pct", "is_urban", "wealth_index",
    "mobile_money_freq", "days_since_last_pmt",
    "payment_regularity", "pmt_vs_schedule", "days_locked"
]

cph = CoxPHFitter()
cph.fit(survival_df[cox_features], duration_col="duration_days", event_col="churned")
cph.print_summary()

fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title("Cox PH Model — Hazard Ratios for Churn", fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / "05_cox_hazard_ratios.png", dpi=150, bbox_inches="tight")
plt.show()

## 5.4 30-Day Churn Classifier

In [ ]:
# ── Label: will churn within 30 days from current snapshot? ──────────────────
# We simulate this as: churned AND days_to_churn within 30 days of observation
survival_df["churn_30d"] = ((survival_df["churned"] == 1) & (survival_df["duration_days"] < 30)).astype(int)

FEATURES_CHURN = [
    "downpayment_pct", "contract_months", "is_urban", "wealth_index",
    "mobile_money_freq", "days_since_last_pmt",
    "payment_regularity", "pmt_vs_schedule", "days_locked"
]

X = survival_df[FEATURES_CHURN]
y = survival_df["churn_30d"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
    eval_metric="logloss", random_state=42, verbosity=0
)
xgb.fit(X_tr, y_tr)
proba = xgb.predict_proba(X_te)[:, 1]
auc   = roc_auc_score(y_te, proba)

print(f"30-day churn classifier AUC-ROC: {auc:.4f}")
print(classification_report(y_te, (proba > 0.5).astype(int)))

## 5.5 Risk Score Output — Field Agent View

In [ ]:
# Generate daily risk scores for all active customers
all_scores = xgb.predict_proba(X)[:, 1]
survival_df["churn_risk_score"] = all_scores.round(4)
survival_df["risk_band"] = pd.cut(
    survival_df["churn_risk_score"],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["Low", "Medium", "High", "Critical"]
)

# Top 20 at-risk customers — this is the daily call centre list
priority_list = (
    survival_df[survival_df["churned"] == 0]  # active customers only
    .sort_values("churn_risk_score", ascending=False)
    .head(20)[["customer_id", "churn_risk_score", "risk_band",
                "days_since_last_pmt", "days_locked", "pmt_vs_schedule"]]
)

print("--- Daily Priority List for Field Agents ---")
print(priority_list.to_string(index=False))

---

**Key finding:** `days_since_last_payment` and `days_locked` are the strongest real-time churn signals. The survival model shows the median customer-at-risk churns by day 180 — providing a clear intervention window for field agents.

**Next:** `06_integrated_dashboard.ipynb` — tie all three modules into a unified decision view.